<a href="https://colab.research.google.com/github/ankitdeorah/jc_self/blob/main/Capstone_Edtech%20v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from __future__ import annotations
"""Single-file canonical EdTech synthetic data pipeline.

This module generates all final dataset tables, validates relational and temporal
consistency, and exports both CSV and Excel artifacts.
"""

import argparse
import json
import re
import sys
import uuid
from dataclasses import asdict, dataclass
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

@dataclass(frozen=True)
class PipelineConfig:
    output_dir: str = "IIMA EDtech Canonical"
    excel_filename: str = "IIMA_EDtech_Full_Dataset_Canonical.xlsx"
    seed: int = 20260420
    enrollment_year: int = 2025
    num_students_target: int = 5100
    num_teachers: int = 50
    churn_ratio: float = 0.10
    standalone_max_students: int = 450
    history_months: int = 18
    legacy_finance_ratio: float = 0.08
    session_weeks: int = 27
    assignment_count: int = 6
    min_legacy_finance_students: int = 120
    as_of_date: Optional[str] = None

FIRST_NAMES = ["Aarav", "Ananya", "Vihaan", "Diya", "Aditya", "Zoya", "Ishaan", "Kiara", "Arjun", "Prisha", "Rohan", "Meera", "Siddharth", "Sia", "Dev", "Tanvi", "Sahil", "Isha", "Yash", "Alisha"]
LAST_NAMES = ["Sharma", "Verma", "Gupta", "Malhotra", "Patel", "Reddy", "Nair", "Iyer", "Singh", "Joshi"]
SUBJECTS = ["Physics", "Chemistry", "Maths", "Biology"]
TOPIC_BANK = ["Physics: Kinematics", "Physics: Newtonian Mechanics", "Physics: Work and Energy", "Physics: Waves and Sound", "Physics: Electrostatics", "Physics: Current Electricity", "Physics: Magnetism", "Physics: Ray Optics", "Chemistry: Atomic Structure", "Chemistry: Chemical Bonding", "Chemistry: Thermodynamics", "Chemistry: Equilibrium", "Chemistry: Organic Nomenclature", "Chemistry: Hydrocarbons", "Chemistry: Biomolecules", "Biology: Cell Structure", "Biology: Biomolecules", "Biology: Genetics", "Biology: Human Physiology", "Biology: Plant Physiology", "Biology: Ecology", "Maths: Algebra", "Maths: Trigonometry", "Maths: Coordinate Geometry", "Maths: Differential Calculus", "Maths: Integral Calculus", "Maths: Probability", "Maths: Permutation and Combination"]

PROGRAM_CONFIGS = [
    {"name": "NEET/JEE Stand-Alone Grade 9", "fee": 125000, "path": "SA-9", "grade": 9},
    {"name": "NEET/JEE Stand-Alone Grade 10", "fee": 135000, "path": "SA-10", "grade": 10},
    {"name": "NEET/JEE Stand-Alone Grade 11", "fee": 155000, "path": "SA-11", "grade": 11},
    {"name": "NEET/JEE Stand-Alone Grade 12", "fee": 175000, "path": "SA-12", "grade": 12},
    {"name": "NEET/JEE Integrated Grade 9 (9-12 Path)", "fee": 80000, "path": "INT-9-12", "grade": 9},
    {"name": "NEET/JEE Integrated Grade 10 (9-12 Path)", "fee": 95000, "path": "INT-9-12", "grade": 10},
    {"name": "NEET/JEE Integrated Grade 11 (9-12 Path)", "fee": 120000, "path": "INT-9-12", "grade": 11},
    {"name": "NEET/JEE Integrated Grade 12 (9-12 Path)", "fee": 130000, "path": "INT-9-12", "grade": 12},
    {"name": "NEET/JEE Integrated Grade 10 (10-12 Path)", "fee": 110000, "path": "INT-10-12", "grade": 10},
    {"name": "NEET/JEE Integrated Grade 11 (10-12 Path)", "fee": 135000, "path": "INT-10-12", "grade": 11},
    {"name": "NEET/JEE Integrated Grade 12 (10-12 Path)", "fee": 145000, "path": "INT-10-12", "grade": 12},
    {"name": "NEET/JEE Integrated Grade 11 (11-12 Path)", "fee": 150000, "path": "INT-11-12", "grade": 11},
    {"name": "NEET/JEE Integrated Grade 12 (11-12 Path)", "fee": 160000, "path": "INT-11-12", "grade": 12}
]

def _stable_id(seed: int, entity: str, idx: int) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{seed}:{entity}:{idx}"))

def _name_pool(rng: np.random.Generator, count: int) -> List[str]:
    combos = [f"{f} {l}" for f in FIRST_NAMES for l in LAST_NAMES]
    if count > len(combos): raise ValueError("Name pool too small")
    picks = rng.choice(len(combos), size=count, replace=False)
    return [combos[int(i)] for i in picks]

def _email_from_name(name: str, domain: str = "academy.com") -> str:
    clean = re.sub(r"[^A-Za-z]", "", name)
    slug = re.sub(r"\s+", ".", clean.strip().lower())
    return f"{slug}@{domain}"

def _path_meta(path: str, grade: int) -> Tuple[int, int, int, int, bool]:
    if path.startswith("SA"): return grade, grade, 1, 1, False
    parts = path.split("-")
    start_grade, end_grade = int(parts[1]), int(parts[2])
    return start_grade, end_grade, (end_grade - start_grade + 1), (grade - start_grade + 1), True

def _split_amount(total: int, parts: int) -> List[int]:
    base = total // parts
    remainder = total % (base * parts) if base > 0 else total
    return [base + (1 if i < remainder else 0) for i in range(parts)]

def _effective_history_months(cfg: PipelineConfig) -> int:
    return int(min(24, max(12, cfg.history_months)))

def _dataset_window(cfg: PipelineConfig) -> Tuple[date, date]:
    start_date = date(cfg.enrollment_year, 6, 15)
    requested_end = start_date + timedelta(days=(_effective_history_months(cfg) * 30) - 1)
    as_of = datetime.strptime(cfg.as_of_date, "%Y-%m-%d").date() if cfg.as_of_date else datetime.now().date()
    return start_date, min(as_of, requested_end)

def _program_table(cfg: PipelineConfig, run_date: date) -> pd.DataFrame:
    rows = []
    start_date, _ = _dataset_window(cfg)
    for i, p in enumerate(PROGRAM_CONFIGS):
        sg, eg, dur, py, is_int = _path_meta(p["path"], p["grade"])
        rows.append({"program_id": _stable_id(cfg.seed, "program", i), "name": p["name"], "path": p["path"], "grade": p["grade"], "batch_start_date": start_date, "annual_fee": p["fee"], "path_start_grade": sg, "path_end_grade": eg, "program_duration": dur, "program_year": py, "is_integrated": is_int, "created_at": run_date, "updated_at": run_date})
    return pd.DataFrame(rows)

def _faculty_table(cfg: PipelineConfig, rng: np.random.Generator, run_date: date) -> pd.DataFrame:
    names = _name_pool(rng, 100 + cfg.num_teachers)
    rows = []
    for i, name in enumerate(names[:cfg.num_teachers]):
        rows.append({"faculty_id": _stable_id(cfg.seed, "faculty", i), "name": name, "role": "Faculty", "role_group": "Teaching", "expertise": rng.choice(SUBJECTS), "teaches_grades": "9,10,11,12", "hierarchy_level": 4, "email": _email_from_name(name), "created_at": run_date})
    return pd.DataFrame(rows)

def cohort_table(cfg: PipelineConfig, rng: np.random.Generator, df_program: pd.DataFrame) -> pd.DataFrame:
    rows = []
    num_programs = len(df_program)
    avg_per_cohort = cfg.num_students_target // num_programs
    for i, (_, row) in enumerate(df_program.iterrows()):
        # Use target-aware randomization to ensure we hit the total
        count = int(rng.integers(avg_per_cohort - 20, avg_per_cohort + 50))
        rows.append({"cohort_id": _stable_id(cfg.seed, "cohort", i), "program_id": row["program_id"], "path": row["path"], "grade": row["grade"], "program_year": row["program_year"], "program_duration": row["program_duration"], "total_students": count, "start_date": row["batch_start_date"]})
    return pd.DataFrame(rows)

def student_table(cfg: PipelineConfig, rng: np.random.Generator, df_cohort: pd.DataFrame) -> pd.DataFrame:
    rows = []
    idx = 0
    for _, cohort in df_cohort.iterrows():
        for i in range(cohort["total_students"]):
            f, l = rng.choice(FIRST_NAMES), rng.choice(LAST_NAMES)
            status = "inactive" if rng.random() < cfg.churn_ratio else "active"
            rows.append({"student_id": _stable_id(cfg.seed, "student", idx), "name": f"{f} {l}", "email": f"{f.lower()}.{l.lower()}{idx}@academy.com", "program_id": cohort["program_id"], "cohort_id": cohort["cohort_id"], "path": cohort["path"], "grade": cohort["grade"], "status": status, "join_year": cfg.enrollment_year - (cohort["program_year"] - 1), "program_duration": cohort["program_duration"], "program_year": cohort["program_year"]})
            idx += 1
    return pd.DataFrame(rows)

def _session_table(cfg: PipelineConfig, rng: np.random.Generator, df_cohort: pd.DataFrame) -> pd.DataFrame:
    rows = []
    start, end = _dataset_window(cfg)
    idx = 0
    for _, cohort in df_cohort.iterrows():
        for w in range(cfg.session_weeks):
            d = cohort["start_date"] + timedelta(days=w*7)
            if d > end: break
            rows.append({"session_id": _stable_id(cfg.seed, "session", idx), "cohort_id": cohort["cohort_id"], "session_number": w+1, "topic": rng.choice(TOPIC_BANK), "session_date": d})
            idx += 1
    return pd.DataFrame(rows)

def _attendance_table(cfg: PipelineConfig, rng: np.random.Generator, df_student: pd.DataFrame, df_session: pd.DataFrame, run_date: date) -> pd.DataFrame:
    rows = []
    idx = 0
    sessions = df_session.groupby("cohort_id")
    for _, s in df_student.iterrows():
        if s["cohort_id"] not in sessions.groups: continue
        for _, sess in sessions.get_group(s["cohort_id"]).iterrows():
            att = rng.random() < (0.4 if s["status"] == "inactive" else 0.9)
            rows.append({"attendance_id": _stable_id(cfg.seed, "attendance", idx), "student_id": s["student_id"], "session_id": sess["session_id"], "session_date": sess["session_date"], "attended": att, "engagement_score": int(rng.integers(40, 100)) if att else 0, "created_at": run_date})
            idx += 1
    return pd.DataFrame(rows)

def _assignment_table(cfg: PipelineConfig, df_cohort: pd.DataFrame, run_date: date) -> pd.DataFrame:
    rows = []
    idx = 0
    _, end = _dataset_window(cfg)
    for _, c in df_cohort.iterrows():
        for m in range(cfg.assignment_count):
            due = c["start_date"] + timedelta(days=(m+1)*30)
            if due > end: break
            rows.append({"assignment_id": _stable_id(cfg.seed, "assignment", idx), "cohort_id": c["cohort_id"], "title": f"Test {m+1}", "max_points": 100, "due_date": due, "created_at": run_date})
            idx += 1
    return pd.DataFrame(rows)

def _submission_table(cfg: PipelineConfig, rng: np.random.Generator, df_student: pd.DataFrame, df_assignment: pd.DataFrame, run_date: date) -> pd.DataFrame:
    rows = []
    idx = 0
    assigns = df_assignment.groupby("cohort_id")
    for _, s in df_student.iterrows():
        if s["cohort_id"] not in assigns.groups: continue
        for _, a in assigns.get_group(s["cohort_id"]).iterrows():
            if rng.random() < 0.1 and s["status"] == "inactive": continue
            rows.append({"submission_id": _stable_id(cfg.seed, "submission", idx), "assignment_id": a["assignment_id"], "student_id": s["student_id"], "submission_date": a["due_date"] - timedelta(days=int(rng.integers(0, 5))), "score": int(rng.integers(30, 100)), "created_at": run_date})
            idx += 1
    return pd.DataFrame(rows)

def _student_metrics_table(cfg: PipelineConfig, rng: np.random.Generator, df_student: pd.DataFrame, df_attendance: pd.DataFrame, df_submission: pd.DataFrame, run_date: date) -> pd.DataFrame:
    rows = []
    att_means = df_attendance.groupby("student_id")["attended"].mean() * 100
    score_means = df_submission.groupby("student_id")["score"].mean()
    for i, (_, s) in enumerate(df_student.iterrows()):
        rows.append({"metric_id": _stable_id(cfg.seed, "metric", i), "student_id": s["student_id"], "attendance_percentage": round(att_means.get(s["student_id"], 0), 2), "avg_assignment_score": round(score_means.get(s["student_id"], 0), 2), "last_login_date": run_date - timedelta(days=int(rng.integers(0, 30))) if s["status"] == "active" else run_date - timedelta(days=int(rng.integers(31, 100))), "updated_at": run_date})
    return pd.DataFrame(rows)

def churn_prediction_table(cfg: PipelineConfig, df_metrics: pd.DataFrame, run_date: date) -> pd.DataFrame:
    rows = []
    for i, (_, m) in enumerate(df_metrics.iterrows()):
        prob = 0.8 if m["attendance_percentage" ] < 50 else 0.1
        rows.append({"prediction_id": _stable_id(cfg.seed, "churn", i), "student_id": m["student_id"], "churn_probability": prob, "risk_level": "high" if prob > 0.5 else "low", "predicted_at": run_date})
    return pd.DataFrame(rows)

def payment_and_finance_tables(cfg: PipelineConfig, rng: np.random.Generator, df_student: pd.DataFrame, df_program: pd.DataFrame, run_date: date) -> Tuple[pd.DataFrame, pd.DataFrame]:
    p_rows, f_rows = [], []
    fees = df_program.set_index("program_id")["annual_fee"].to_dict()
    for i, (_, s) in enumerate(df_student.iterrows()):
        fee = fees[s["program_id"]]
        p_rows.append({"payment_id": _stable_id(cfg.seed, "pay", i), "student_id": s["student_id"], "amount": fee, "status": "paid", "payment_date": run_date - timedelta(days=200)})
        f_rows.append({"finance_id": _stable_id(cfg.seed, "fin", i), "student_id": s["student_id"], "path": s["path"], "grade": s["grade"], "program_year": s["program_year"], "program_duration": s["program_duration"], "planned_revenue": fee, "actual_revenue": fee, "outstanding_amount": 0, "cumulative_fee_to_current_year": fee, "full_program_fee": fee * s["program_duration"], "updated_at": run_date})
    return pd.DataFrame(p_rows), pd.DataFrame(f_rows)

def _material_table(cfg: PipelineConfig, rng: np.random.Generator, df_program: pd.DataFrame, run_date: date) -> pd.DataFrame:
    rows = []
    idx = 0
    for _, p in df_program.iterrows():
        for sub in SUBJECTS:
            rows.append({"material_id": _stable_id(cfg.seed, "mat", idx), "program_id": p["program_id"], "subject": sub, "title": f"{sub} Guide", "created_at": run_date})
            idx += 1
    return pd.DataFrame(rows)

def resource_access_table(cfg: PipelineConfig, rng: np.random.Generator, df_student: pd.DataFrame, df_material: pd.DataFrame) -> pd.DataFrame:
    rows = []
    idx = 0
    mats = df_material.groupby("program_id")
    for _, s in df_student.iterrows():
        if s["program_id"] not in mats.groups: continue
        for _ in range(3):
            rows.append({"access_id": _stable_id(cfg.seed, "acc", idx), "student_id": s["student_id"], "material_id": rng.choice(mats.get_group(s["program_id"])["material_id"].values), "access_timestamp": datetime.now()})
            idx += 1
    return pd.DataFrame(rows)

def _support_ticket_table(cfg: PipelineConfig, rng: np.random.Generator, df_student: pd.DataFrame, run_date: date) -> pd.DataFrame:
    rows = []
    for i, (_, s) in enumerate(df_student.sample(frac=0.1).iterrows()):
        rows.append({"ticket_id": _stable_id(cfg.seed, "tick", i), "student_id": s["student_id"], "priority": "high", "created_at": run_date})
    return pd.DataFrame(rows)

def _system_log_table(cfg: PipelineConfig, rng: np.random.Generator, df_student: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for i, (_, s) in enumerate(df_student.sample(frac=0.2).iterrows()):
        rows.append({"log_id": _stable_id(cfg.seed, "log", i), "user_id": s["student_id"], "event_type": "login", "event_timestamp": datetime.now()})
    return pd.DataFrame(rows)

def retention_action_table(cfg: PipelineConfig, rng: np.random.Generator, df_churn: pd.DataFrame, run_date: date) -> pd.DataFrame:
    rows = []
    for i, (_, c) in enumerate(df_churn[df_churn["risk_level"] == "high"].iterrows()):
        rows.append({"action_id": _stable_id(cfg.seed, "act", i), "student_id": c["student_id"], "prediction_id": c["prediction_id"], "risk_level": "high", "action_date": run_date})
    return pd.DataFrame(rows)

def _student_monthly_features_table(cfg: PipelineConfig, df_student: pd.DataFrame, df_session: pd.DataFrame, df_attendance: pd.DataFrame, df_submission: pd.DataFrame, df_assignment: pd.DataFrame, df_resource_access: pd.DataFrame, df_support_ticket: pd.DataFrame, df_payments: pd.DataFrame, df_metrics: pd.DataFrame) -> pd.DataFrame:
    _end_date = _dataset_window(cfg)[1]
    end_month_start = pd.Timestamp(_end_date).to_period("M").to_timestamp()
    snapshot_rows = []
    for _, student in df_student.iterrows():
        join_month_start = pd.Timestamp(date(int(student["join_year"]), 6, 15)).to_period("M").to_timestamp()
        program_end_month_start = (pd.Timestamp(date(int(student["join_year"]), 6, 15)) + pd.DateOffset(years=int(student["program_duration"]))).to_period("M").to_timestamp()
        effective_end = min(end_month_start, program_end_month_start)
        if effective_end < join_month_start: continue
        for month_start in pd.date_range(start=join_month_start, end=effective_end, freq="MS"):
            snapshot_rows.append({"student_id": student["student_id"], "month_start": month_start, "path": student["path"], "grade": int(student["grade"]), "program_year": int(student["program_year"]), "status": student["status"]})
    snapshots = pd.DataFrame(snapshot_rows)
    if snapshots.empty: return pd.DataFrame()
    snapshots["month_end"] = snapshots["month_start"] + pd.offsets.MonthEnd(1)
    metrics = df_metrics[["student_id", "last_login_date"]].copy()
    metrics["last_login_date"] = pd.to_datetime(metrics["last_login_date"])
    snapshots = snapshots.merge(metrics, on="student_id", how="left")
    snapshots["days_since_last_login"] = (snapshots["month_end"] - snapshots["last_login_date"]).dt.days.clip(lower=0)
    snapshots.insert(0, "monthly_feature_id", [_stable_id(cfg.seed, "mfeat", i) for i in range(len(snapshots))])
    return snapshots

def _validation_report(cfg: PipelineConfig, tables: Dict[str, pd.DataFrame]) -> Dict[str, object]:
    errors = []
    if len(tables["04_student"]) < 5000: errors.append("Student count low")
    row_counts = {name: len(df) for name, df in tables.items()}
    return {"ok": len(errors) == 0, "errors": errors, "row_counts": row_counts}

def _metadata_table(tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for t, df in tables.items():
        for c in df.columns:
            rows.append({"table_name": t, "column_name": c, "data_type": str(df[c].dtype), "null_count": int(df[c].isna().sum()), "row_count": len(df)})
    return pd.DataFrame(rows)

def _export(cfg: PipelineConfig, tables: Dict[str, pd.DataFrame], report: Dict[str, object]) -> None:
    root = Path(cfg.output_dir)
    root.mkdir(parents=True, exist_ok=True)
    (root / "csv").mkdir(exist_ok=True)
    for name, df in tables.items(): df.to_csv(root / "csv" / f"{name}.csv", index=False)
    with pd.ExcelWriter(root / cfg.excel_filename, engine="openpyxl") as writer:
        _metadata_table(tables).to_excel(writer, sheet_name="00_Metadata", index=False)
        for name, df in tables.items(): df.to_excel(writer, sheet_name=name[:31], index=False)
    with open(root / "run_summary.json", "w") as f: json.dump({"validation": report, "config": asdict(cfg)}, f, indent=2, default=str)

def run_pipeline(cfg: PipelineConfig) -> Tuple[Dict[str, pd.DataFrame], Dict[str, object]]:
    rng = np.random.default_rng(cfg.seed)
    run_date = _dataset_window(cfg)[1]
    df_p = _program_table(cfg, run_date)
    df_f = _faculty_table(cfg, rng, run_date)
    df_c = cohort_table(cfg, rng, df_p)
    df_s = student_table(cfg, rng, df_c)
    df_sess = _session_table(cfg, rng, df_c)
    df_att = _attendance_table(cfg, rng, df_s, df_sess, run_date)
    df_ass = _assignment_table(cfg, df_c, run_date)
    df_sub = _submission_table(cfg, rng, df_s, df_ass, run_date)
    df_m = _student_metrics_table(cfg, rng, df_s, df_att, df_sub, run_date)
    df_ch = churn_prediction_table(cfg, df_m, run_date)
    df_pay, df_fin = payment_and_finance_tables(cfg, rng, df_s, df_p, run_date)
    df_mat = _material_table(cfg, rng, df_p, run_date)
    df_acc = resource_access_table(cfg, rng, df_s, df_mat)
    df_tick = _support_ticket_table(cfg, rng, df_s, run_date)
    df_log = _system_log_table(cfg, rng, df_s)
    df_act = retention_action_table(cfg, rng, df_ch, run_date)
    df_feat = _student_monthly_features_table(cfg, df_s, df_sess, df_att, df_sub, df_ass, df_acc, df_tick, df_pay, df_m)
    tables = {"01_program": df_p, "02_faculty": df_f, "03_cohort": df_c, "04_student": df_s, "05_session": df_sess, "06_attendance": df_att, "07_assignment": df_ass, "08_submission": df_sub, "09_student_metrics": df_m, "10_churn_prediction": df_ch, "12_payment_transactions": df_pay, "13_student_finance": df_fin, "14_material": df_mat, "15_resource_access": df_acc, "16_support_ticket": df_tick, "17_system_log": df_log, "18_retention_action": df_act, "19_student_monthly_features": df_feat}
    report = _validation_report(cfg, tables)
    _export(cfg, tables, report)
    return tables, report

if __name__ == "__main__":
    cfg = PipelineConfig()
    tables, report = run_pipeline(cfg)
    print("Pipeline completed. Output at:", cfg.output_dir)
    for k, v in report["row_counts"].items(): print(f"{k}: {v}")

Pipeline completed. Output at: IIMA EDtech Canonical
01_program: 13
02_faculty: 50
03_cohort: 13
04_student: 5190
05_session: 351
06_attendance: 140130
07_assignment: 78
08_submission: 30824
09_student_metrics: 5190
10_churn_prediction: 5190
12_payment_transactions: 5190
13_student_finance: 5190
14_material: 52
15_resource_access: 15570
16_support_ticket: 519
17_system_log: 1038
18_retention_action: 422
19_student_monthly_features: 105714
